In [ ]:
import pandas as pd
import numpy as np
import re
import networkx as nx
from matplotlib.lines import Line2D
import os
import cstarpy.inference
import cstarpy.integration
from cstarpy.preprocessing import PerturbationMagnitude
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt
import pylab as plt
import openpyxl


In [ ]:
# Load in data

SZABO_FILE = "BTLAvsTCR_24h.csv"
COMPOUND_FILE  = "cd8_limma_merged_filtered_targets_ic50.csv"

cd8_szabo = pd.read_csv(SZABO_FILE)
cd8_szabo["gene"] = cd8_szabo["gene"].astype(str).str.strip()   # in case of trailing whitespace
cd8_szabo = cd8_szabo.set_index("gene")



# 2. Load compound file + average replicates + zero insignificant

df = pd.read_csv(COMPOUND_FILE)

df_avg = (
    df.groupby(["compound_name", "gene"])
    .agg(logFC=("logFC", "mean"), adj_P_Val=("adj.P.Val", "mean"))
    .reset_index()
)

df_avg["logFC_threshold"] = np.where(
    df_avg["adj_P_Val"] < 0.05, df_avg["logFC"], 0
)

# 3. Pivot: compounds × genes 

pivot = df_avg.pivot_table(
    index   = "compound_name",
    columns = "gene",
    values  = "logFC_threshold",
    aggfunc = "first"
).fillna(0)



In [ ]:
cd8_szabo

In [ ]:
# 4. Align to Szabo reference vector (raw) 

REF_COL = "log2FoldChange"

common_genes  = pivot.columns.intersection(cd8_szabo.index)
pivot_aligned = pivot[common_genes]
szabo_aligned = cd8_szabo.loc[common_genes, REF_COL].values

#  5. DPD per compound = pivot @ szabo (no normalisation)

dpd_vec = pivot_aligned.values @ szabo_aligned

dpd_df = pd.DataFrame({
    "compound_name" : pivot_aligned.index,
    "dpd"           : dpd_vec.round(4),
    "direction"     : ["drives_activation" if x > 0 else "drives_resting"
                       for x in dpd_vec]
}).sort_values("dpd", ascending=False).reset_index(drop=True)

#  6. Add targets and mechanism 

targets = pd.read_csv(COMPOUND_FILE)[
    ["compound_name", "target_protein", "mechanism"]
].drop_duplicates(subset="compound_name")

dpd_df = dpd_df.merge(targets, on="compound_name", how="left")

#  7. Save 
dpd_df.to_csv("dpd_sum_per_compound_raw_btla.csv", index=False)
print("\n✓ Saved dpd_sum_per_compound_raw_btla.csv")


In [ ]:
# BTLA module (separate from DPD) — dot product of each compound's logFC vector
# against a 'stat'-based reference vector from cd8_szabo (BTLAvsTCR_4h.csv),
# mirroring the DPD calc above but using 'stat' instead of 'log2FoldChange',
# zeroed out wherever cd8_szabo's own p-value > 0.05
# NOTE: assumes cd8_szabo has columns 'stat' and 'pvalue' (DESeq2-style output) —
# update STAT_COL / PVAL_COL below if the actual column names differ

STAT_COL = "stat"
PVAL_COL = "pvalue"

szabo_stat_threshold = np.where(cd8_szabo[PVAL_COL] > 0.05, 0, cd8_szabo[STAT_COL])
szabo_stat = pd.Series(szabo_stat_threshold, index=cd8_szabo.index)

common_genes_btla  = pivot.columns.intersection(szabo_stat.index)
pivot_aligned_btla = pivot[common_genes_btla]
szabo_aligned_btla = szabo_stat.loc[common_genes_btla].values

btla_vec = pivot_aligned_btla.values @ szabo_aligned_btla

btla_df = pd.DataFrame({
    "compound_name" : pivot_aligned_btla.index,
    "btla"          : btla_vec.round(4),
}).sort_values("btla", ascending=False).reset_index(drop=True)

btla_df.to_csv("btla_deg_per_compound_btla.csv", index=False)
print("\n✓ Saved btla_deg_per_compound_btla.csv")


In [ ]:
# Load in data file

dpd = pd.read_csv("dpd_sum_per_compound_raw_btla.csv")

# Create list of top 4 and bottom 4
top4 = dpd.nlargest(4, "dpd")[["compound_name", "dpd", "target_protein", "mechanism"]]
bottom4 = dpd.nsmallest(4, "dpd")[["compound_name", "dpd", "target_protein", "mechanism"]]

print(top4.to_string(index=False))
print(bottom4.to_string(index=False))

#  Extract their target genes as modules 
selected = pd.concat([top4, bottom4])

modules = set()
for targets in selected["target_protein"].dropna():
    genes = [g.strip().upper() for g in re.split(r"[;,]", str(targets)) if g.strip()]
    modules.update(genes)

In [ ]:
import pandas as pd

#Load in IC50 values

df = pd.read_csv("cd8_limma_merged_filtered_targets_ic50_dpd.csv")

# Filter to top/bottom 4 compounds 
selected_compounds = [
    "Thapsigargin", "QS-11", "Pevonedistat", "navitoclax",
    "ruxolitinib", "Sapanisertib", "Temsirolimus", "BMS-536924"
]

df_selected = df[df["compound_name"].isin(selected_compounds)].copy()

#  Extract IC50 per compound 
ic50_per_compound = df_selected.groupby("compound_name").agg(
    IC50_nM       = ("IC50_nM",       "first"),
    dose_uM       = ("dose_uM",       "first"),
    target_protein = ("target_protein", "first"),
    mechanism     = ("mechanism",     "first")
).reset_index()

# Parse IC50 to float
def parse_ic50(val):
    if pd.isna(val) or str(val).strip().lower() == "unknown":
        return None
    try:
        return round(sum(float(v) for v in str(val).split(",")) /
                     len(str(val).split(",")), 4)
    except:
        return None

ic50_per_compound["IC50_nM_clean"] = ic50_per_compound["IC50_nM"].apply(parse_ic50)
ic50_per_compound["IC50_uM"]       = ic50_per_compound["IC50_nM_clean"] / 1000
ic50_per_compound["g"]             = 1 / (1 + ic50_per_compound["dose_uM"] /
                                          ic50_per_compound["IC50_uM"])

print(ic50_per_compound[["compound_name", "dose_uM", "IC50_nM_clean",
                          "IC50_uM", "g", "target_protein"]].to_string(index=False))

In [ ]:
ic50_per_compound

In [ ]:
import pandas as pd
import numpy as np

compound_to_module = {
    "AT9283":"JAK", "Apicidin":"HDAC", "BI 2536":"PLK", "BMS-536924":"IGF1R",
    "BRD-K76674262":"STAT3", "Bisindolylmaleimide II":"PKC", "Brefeldin A":"ATPase",
    "CGP-60474":"CDK", "CYT-387":"JAK", "Calcitriol":"VDR", "Crizotinib":"ALK",
    "Decitabine":"DNMT", "Deforolimus":"MTOR", "Dexamethasone":"GR",
    "Dorsomorphin":"AMPK", "EX-527":"SIRT", "Entospletinib":"SYK", "Etomoxir":"CPT1",
    "Forskolin":"ADCY", "Fostamatinib":"SYK", "Geldanamycin":"HSP90", "I-BET 762":"BRD",
    "JNK-9L (JNK Inhibitor)":"JNK", "NFkB Activation Inhibitor II":"NFkB",
    "NVP-AUY922":"HSP90", "NVP-BHG 712":"EPH", "Neratinib":"ERBB", "PHA-793887":"CDK",
    "Penfluridol":"DRD", "Pevonedistat":"NAE", "Pomalidomide":"CRBN", "Ponatinib":"ABL",
    "Purmorphamine":"SMO", "QS-11":"ARFGAP", "Rebastinib":"ABL", "Resveratrol":"LCK",
    "Rucaparib":"PARP", "Sapanisertib":"MTOR", "Selumetinib":"MEK", "Serdemetan":"MDM2",
    "Sorafenib":"RAF", "TG-101348":"JAK", "TWS-119":"GSK3", "Temsirolimus":"MTOR",
    "Thapsigargin":"SERCA", "Tozasertib":"Aurora", "VX-11e":"ERK", "Veliparib":"PARP",
    "Withaferin A":"NFkB", "Wortmannin":"PI3K", "YM-155":"BIRC5", "belinostat":"HDAC",
    "camptothecin":"TOP1", "navitoclax":"BCL", "panobinostat":"HDAC", "ruxolitinib":"JAK",
}


In [ ]:

dpd = pd.read_csv("dpd_sum_per_compound_raw_btla.csv")
dpd_col = "dpd"
dpd["module"] = dpd["compound_name"].map(compound_to_module)
dpd_inh = dpd[dpd["mechanism"] == "Inhibitor"].dropna(subset=["module"]).copy()

# Top 4 + bottom 4 DRUGS → their modules
top4    = dpd_inh.nlargest(4,  dpd_col)
bottom4 = dpd_inh.nsmallest(4, dpd_col)
selected_modules = pd.concat([top4, bottom4])["module"].unique().tolist()

print("Top 4 drugs:");    print(top4[["compound_name", dpd_col, "module"]].to_string(index=False))
print("\nBottom 4 drugs:"); print(bottom4[["compound_name", dpd_col, "module"]].to_string(index=False))
print("\nSelected modules:", selected_modules)

# All inhibitors hitting those modules
drug_set = dpd_inh[dpd_inh["module"].isin(selected_modules)].copy()
print("\nDrugs per module:")
print(drug_set.groupby("module")["compound_name"].apply(list).to_string())

# Build drug-module-IC50 table
df = pd.read_csv("cd8_limma_merged_filtered_targets_ic50_dpd.csv")

def parse_ic50(val):
    if pd.isna(val) or str(val).strip().lower() == "unknown":
        return np.nan
    try:
        v = str(val).replace("/", ",")
        vals = [float(x) for x in v.split(",") if x.strip()]
        return round(np.exp(np.mean(np.log(vals))), 4)
    except Exception:
        return np.nan

rows = []
for _, r in drug_set.iterrows():
    compound = r["compound_name"]
    ic50_raw = df[df["compound_name"] == compound]["IC50_nM"].iloc[0] \
               if (df["compound_name"] == compound).any() else np.nan
    rows.append({"compound_name": compound, "module": r["module"],
                 "IC50_nM": parse_ic50(ic50_raw), "dpd": r[dpd_col],
                 "mechanism": r["mechanism"]})

drug_module_ic50 = pd.DataFrame(rows).sort_values("module")
drug_module_ic50.to_csv("top4_bottom4_selected_modules_drugs_ic50_btla.csv", index=False)
print("\n✓ Saved top4_bottom4_selected_modules_drugs_ic50_btla.csv")

In [ ]:
IC50_df = pd.read_csv("top4_bottom4_selected_modules_drugs_ic50_btla.csv")\
    .rename(columns={"module": "Module", "compound_name": "Drug", "IC50_nM": "IC50"})
IC50_df["IC50"] = IC50_df["IC50"] / 1000          
IC50_df = IC50_df.set_index("Module").drop_duplicates()
IC50_df_reset = IC50_df.reset_index()

modules = IC50_df.index.unique().tolist()
drugs   = IC50_df_reset["Drug"].drop_duplicates().tolist()
print(f"Modules ({len(modules)}): {modules}")
print(f"Drugs   ({len(drugs)}): {drugs}")

# Dose and mechanism per compound
dose_info = df.groupby("compound_name")["dose_uM"].first()
mech_info = df.groupby("compound_name")["mechanism"].first()

GAMMA = 1.0   # activation coefficient, applied only where mechanism == "Activator"

# Build matrices
inhib_conc_matrix = np.zeros((len(modules), len(drugs)))
ic50_matrix       = np.zeros((len(modules), len(drugs)))
gamma_matrix      = np.zeros((len(modules), len(drugs)))   # 0 = inhibitor, >0 = activator

for i, module in enumerate(modules):
    drugs_for_module = IC50_df_reset["Drug"][IC50_df_reset["Module"] == module].tolist()
    for drug in drugs_for_module:
        ic50 = IC50_df_reset["IC50"][IC50_df_reset["Drug"] == drug].values
        dose = dose_info.get(drug, np.nan)
        if ic50.size == 0 or pd.isna(dose) or drug not in drugs:
            continue
        j = drugs.index(drug)
        inhib_conc_matrix[i, j] = dose
        ic50_matrix[i, j]       = ic50[0]
        gamma_matrix[i, j]      = GAMMA if mech_info.get(drug, "Inhibitor") == "Activator" else 0.0

inhib_conc_df = pd.DataFrame(inhib_conc_matrix, index=modules, columns=drugs)
ic50_df_out   = pd.DataFrame(ic50_matrix,       index=modules, columns=drugs)
pert_df       = pd.DataFrame(np.where(inhib_conc_matrix != 0, 1, 0),
                             index=modules, columns=drugs)

g_matrix = np.ones((len(modules), len(drugs)))

for i in range(len(modules)):
    for j in range(len(drugs)):
        ic50 = ic50_matrix[i, j]
        dose = inhib_conc_matrix[i, j]

        if ic50 <= 0 or dose == 0:
            g_matrix[i, j] = 1.0          # no perturbation g = 1
            continue

        dratio = dose / ic50

        if gamma_matrix[i, j] > 0:
            g_matrix[i, j] = (1 + GAMMA * dratio) / (1 + dratio)   # activator: (1 + gamma*I/R) / (1 + k/IC50)
        else:
            g_matrix[i, j] = 1 / (1 + dratio)                  # inhibitor: 1/(1 + k/IC50)

g_df = pd.DataFrame(g_matrix, index=modules, columns=drugs)
print("\ng matrix (per mechanism):")
print(g_df.round(3).to_string())

# Save (unchanged filenames — keep the _min2 / _top4_bottom4 suffix already in this cell)
inhib_conc_df.to_csv("inhib_conc_matrix_top4_bottom4_btla.csv")   # or _min2.csv in that cell
ic50_df_out.to_csv("ic50_matrix_top4_bottom4_btla.csv")            # or _min2.csv in that cell
pert_df.to_csv("pert_matrix_top4_bottom4_btla.csv")                # or _min2.csv in that cell
g_df.to_csv("g_matrix_top4_bottom4_btla.csv")                      # or _min2.csv in that cell
print("\n✓ Saved inhib_conc, ic50, pert, g matrices")

In [ ]:

COMPOUND_FILE = "cd8_limma_merged_filtered_targets_ic50.csv"
SELECTION     = "top4_bottom4_selected_modules_drugs_ic50_btla.csv"   # from prep notebook
out_dir       = "02_outputs"
os.makedirs(out_dir, exist_ok=True)

# Load the collapsed-module selection built in prep (module column already exists)
drug_gene = pd.read_csv(SELECTION).rename(columns={"compound_name": "drug"})

modules  = drug_gene["module"].drop_duplicates().tolist()
exp_list = drug_gene["drug"].drop_duplicates().tolist()

print(f"Modules ({len(modules)}): {modules}")
print(f"Experiments ({len(exp_list)}): {exp_list}")

In [ ]:
df = pd.read_csv(COMPOUND_FILE)

df_avg = (
    df.groupby(["compound_name", "gene"])
    .agg(logFC=("logFC", "mean"), adj_P_Val=("adj.P.Val", "mean"))
    .reset_index()
)
df_avg["logFC_thresh"] = np.where(df_avg["adj_P_Val"] < 0.05, df_avg["logFC"], 0)

x_df = df_avg.pivot_table(
    index="gene", columns="compound_name", values="logFC_thresh", aggfunc="first"
).fillna(0)[exp_list]

genes = x_df.index.tolist()
x     = x_df.values
print(f"x (expression) shape: {x.shape}  (genes × experiments)")

In [ ]:
dose_info = df.groupby("compound_name")["dose_uM"].first()
mech_info = df.groupby("compound_name")["mechanism"].first()

inhib_conc_matrix_top4_bottom4_btla = np.zeros((len(modules), len(exp_list)))
ic50_matrix_top4_bottom4_btla       = np.ones((len(modules), len(exp_list))) * np.inf   # inf → g=1
gamma_matrix_top4_bottom4_btla      = np.zeros((len(modules), len(exp_list)))            # activation coefficient (gamma)
is_activator_top4_bottom4_btla      = np.zeros((len(modules), len(exp_list)), dtype=bool)  # mechanism flag
valid_matrix_top4_bottom4_btla      = np.zeros((len(modules), len(exp_list)), dtype=bool)  # has real dose+IC50 data


GAMMA = 1.0   # activation coefficient, applied only where mechanism == "Activator"
IC50_OVERRIDES_NM = {
    "Dorsomorphin": 234.6,
    "Apicidin": 0.7,
    "Serdemetan": 240.0,
}

for _, row in drug_gene.iterrows():
    if row["module"] in modules and row["drug"] in exp_list:
        i = modules.index(row["module"])
        j = exp_list.index(row["drug"])
        dose = dose_info.get(row["drug"], np.nan)

        if row["drug"] in IC50_OVERRIDES_NM:
            ic50_nm = IC50_OVERRIDES_NM[row["drug"]]
        else:
            ic50_raw = df[df["compound_name"] == row["drug"]]["IC50_nM"]
            ic50_nm = parse_ic50(ic50_raw.iloc[0]) if len(ic50_raw) else np.nan

        ic50 = ic50_nm / 1000 if pd.notna(ic50_nm) else np.nan
        mech = mech_info.get(row["drug"], "Inhibitor")

        if pd.notna(dose) and pd.notna(ic50):
            inhib_conc_matrix_top4_bottom4_btla[i, j] = dose
            ic50_matrix_top4_bottom4_btla[i, j]       = ic50
            gamma_matrix_top4_bottom4_btla[i, j]      = GAMMA if mech == "Activator" else 0.0
            valid_matrix_top4_bottom4_btla[i, j]      = True
            is_activator_top4_bottom4_btla[i, j]     = (mech == "Activator")
        else:
            print(f"WARNING: missing dose/IC50 for module={row['module']} drug={row['drug']} "
                  f"— excluded from fit (not treated as 'no effect')")

# y_true = (1 + gamma_matrix * dose/IC50) / (1 + dose/IC50)   [gamma=0 collapses to inhibitor form]
# Mechanism-specific activity g (bmra_prep gamma convention), I = dose (uM), K = IC50 (uM):
#   inhibitor: g = 1 / (1 + I/K)
#   activator: g = (1 + gamma * I/K) / (1 + I/K)
dratio_top4_bottom4 = inhib_conc_matrix_top4_bottom4_btla / ic50_matrix_top4_bottom4_btla
y_true_top4_bottom4 = np.where(
    valid_matrix_top4_bottom4_btla,
    np.where(
        is_activator_top4_bottom4_btla,
        (1 + gamma_matrix_top4_bottom4_btla * dratio_top4_bottom4) / (1 + dratio_top4_bottom4),  # activator
        1.0 / (1 + dratio_top4_bottom4),                                                    # inhibitor
    ),
    1.0,
)

print(f"y_true (activity g) shape: {y_true_top4_bottom4.shape}")
print(pd.DataFrame(y_true_top4_bottom4, index=modules, columns=exp_list).round(3).to_string())


In [ ]:
pert_df_btla= pd.read_csv("pert_matrix_top4_bottom4_btla.csv", index_col=0)

residuals, a_coeffs = cstarpy.integration.pathway_activity.prediction.predict_coeffs(
    x, y_true_top4_bottom4, pert_df_btla,
    200_000, 10, 10, 10, 100
)

a_coeffs_df_top4_bottom4_btla = pd.DataFrame(a_coeffs, index=modules, columns=genes)
a_coeffs_df_top4_bottom4_btla.to_csv(os.path.join(out_dir, "a_coeffs_df_top4_bottom4_btla.csv"))
print(f"a_coeffs shape: {a_coeffs.shape}")
trh = 0.0001
print("\nGenes representing each module:")
print((abs(a_coeffs_df_top4_bottom4_btla) > trh).sum(axis="columns").to_string())

In [ ]:
a_coeffs_top4_bottom4_btla = a_coeffs_df_top4_bottom4_btla.values

pathway_activity_top4_bottom4_btla = a_coeffs_top4_bottom4_btla @ x
pd.DataFrame(pathway_activity_top4_bottom4_btla, index=modules, columns=exp_list)\
    .to_csv(os.path.join(out_dir, "pathway_activity_top4_bottom4_btla.csv"))

R_global_top4_bottom4_btla = cstarpy.integration.pathway_activity.calc_global_response_from_pathway_activity(
    cstarpy.integration.pathway_activity.calc_pathway_activity(x, a_coeffs_top4_bottom4_btla),
    modules, exp_list
)
R_global_df_top4_bottom4_btla = pd.DataFrame(R_global_top4_bottom4_btla, index=modules, columns=exp_list)
R_global_df_top4_bottom4_btla.to_csv(os.path.join(out_dir, "R_global_core_top4_bottom4_btla.csv"))

print("Global response matrix:")
print(R_global_df_top4_bottom4_btla.round(3).to_string())



In [ ]:
pd.DataFrame(y_true_top4_bottom4,      index=modules, columns=exp_list).to_csv(os.path.join(out_dir, "y_true_top4_bottom4_btla.csv"))
x_df.to_csv(os.path.join(out_dir, "Data_norm.csv"))
print("\n✓ Saved a_coeffs_btla, pathway_activity_btla, R_global_core_btla, y_true_btla, pert_matrix_btla, Data_norm_btla")

In [ ]:
# R_global_annotated = R_global_core.csv  (modules × drugs)
# R_global_DPDonly = DPD response row    (DPD × drugs)
# pert_modules = pert_matrix.csv     (modules × drugs)
# forbidden_connections = built in-code (no Excel sheet)
drug_data = pd.concat([top4, bottom4])[["compound_name", "dpd", "module"]].to_dict(orient="records")



R_global_core_df = pd.read_csv(os.path.join(out_dir, "R_global_core_top4_bottom4_btla.csv"), index_col=0)
R_global_DPD_df = pd.DataFrame(drug_data)

modules_core = R_global_core_df.index.tolist()
modules_DPD = [item["compound_name"] for item in R_global_DPD_df.to_dict(orient="records")]

exp_ids = R_global_core_df.columns.tolist()

#R_global = R_global_df.values
#n_modules = R_global.shape[0]

#print(R_global.shape)
#display(R_global_df)
pert_modules_df = pd.read_csv("pert_matrix_top4_bottom4_btla.csv", index_col=0)

In [ ]:
out_dir = "02_outputs"

G_not_df = pd.DataFrame(False, index=modules_core, columns=modules_core)
np.fill_diagonal(G_not_df.values, True)
r_mean_br = cstarpy.inference.run_mra(
            R_global_core_df.values,
            pert_modules_df.values,
            G_not=G_not_df.values,
            method='ols', #adjust to 'br' if you want to use the Bayesian regression method or ODR for ortogonal distance regression
            pvalue_threshold=0.05)

r_mean_br_df = pd.DataFrame(r_mean_br, index = modules_core, columns = modules_core)
np.fill_diagonal(r_mean_br_df.values, -1) 
r_mean_br_df

In [ ]:
rp = PerturbationMagnitude(R=R_global_core_df, pert=pert_modules_df, r=r_mean_br_df)
rp.estimate(alpha=1.0)

n_iter = 1200

rps_diff, r_mean, rp_final= cstarpy.inference.run_mra_rp(
            R_global_core_df,
            pert_modules_df,
            G_not=G_not_df.values,
            method='ols', #adjust to 'br' if you want to use the Bayesian regression method or ODR for ortogonal distance regression
            pvalue_threshold=0.05, # adjust p-value if to include more edges in the network
            alpha=10.0,
            rp_init=rp,
            rp_relax=0.005,
            show_progress_bar=True,
            n_iter=n_iter)

r_mean_df = pd.DataFrame(r_mean, index = modules_core, columns = modules_core)
np.fill_diagonal(r_mean_df.values, -1) 

In [ ]:
R_global_DPD_df

In [ ]:
r_mean_df = pd.DataFrame(r_mean, index = modules_core, columns = modules_core)
np.fill_diagonal(r_mean_df.values, -1) 

In [ ]:
import pylab as plt
plt.plot(np.arange(n_iter),rps_diff)
plt.xlabel('Number of iterations')
plt.ylabel('|rp_old - rp_new|/|rp|')
plt.show()

In [ ]:
dpd_by_drug = (
    pd.read_csv("top4_bottom4_selected_modules_drugs_ic50_btla.csv")
    .set_index("compound_name")["dpd"]
)
btla_by_drug = (
    pd.read_csv("btla_deg_per_compound_btla.csv")
    .set_index("compound_name")["btla"]
)

R_global_core_df = R_global_core_df.reindex(columns=exp_ids)
pert_modules_df  = pert_modules_df.reindex(columns=exp_ids)
R_global_DPD_df  = pd.DataFrame(
    [dpd_by_drug.reindex(exp_ids), btla_by_drug.reindex(exp_ids)],
    index=["DPD", "BTLA"]
)

G_not_fptu = pd.DataFrame(False, index=R_global_DPD_df.index, columns=R_global_core_df.index)

r_fptu = cstarpy.inference.run_regularized_mra_fptu(
    data_perturbed=R_global_core_df,
    data_unperturbed=R_global_DPD_df,
    pert=pert_modules_df.values,
    G_not=G_not_fptu.values,
    method='ols',
    threshold=0.1,
    pvalue_threshold=0.4,
    progress_bar='off')

r_fptu_df = pd.DataFrame(r_fptu, index=["DPD", "BTLA"], columns=modules_core)
print("Modules → DPD:")
print(r_fptu_df.loc["DPD"].sort_values().round(3).to_string())
print("\nModules → BTLA:")
print(r_fptu_df.loc["BTLA"].sort_values().round(3).to_string())


In [ ]:
drug_data

In [ ]:
print(R_global_DPD_df.shape, R_global_DPD_df.index.tolist())
print(R_global_core_df.shape, R_global_core_df.index.tolist())
print(G_not_fptu.shape)   # you assume (1, 8)

In [ ]:
nodes = modules_core + ["DPD", "BTLA"]
r_total_mean = pd.DataFrame(np.zeros((len(nodes), len(nodes))), index=nodes, columns=nodes)
r_total_mean.loc[modules_core, modules_core] = r_mean
r_total_mean.loc["DPD", modules_core] = r_fptu_df.loc["DPD", modules_core].values
r_total_mean.loc["BTLA", modules_core] = r_fptu_df.loc["BTLA", modules_core].values
np.fill_diagonal(r_total_mean.values, -1)
r_total_mean.to_csv(os.path.join(out_dir, "r_mean_ols_rp_fptu_prefinal_btla.csv"))
rm_minus_inv = pd.DataFrame(np.linalg.pinv(r_total_mean.values), index=nodes, columns=nodes) * (-1)
rm_minus_inv.to_csv(os.path.join(out_dir, "r_minv_ols_rp_fptu_prefinal_btla.csv"))


In [ ]:
print("Connection matrix r (modules + DPD):")
print(r_total_mean.round(3).to_string())

print("\nMinus inverse −r⁻¹ (modules + DPD):")
print(rm_minus_inv.round(3).to_string())

In [ ]:
R_with_dpd = pd.concat([R_global_core_df, R_global_DPD_df])
pert_full  = pert_modules_df.reindex(index=nodes, fill_value=0)   

r_I = R_with_dpd.reindex(index=nodes).fillna(0) * pert_full
R_I_pred = pd.DataFrame(
    rm_minus_inv.values @ r_I.values,
    index=nodes, columns=exp_ids,
)
print("Predicted systems-level response R_I = -r⁻¹ · r_I:")
print(R_I_pred.round(3).to_string())
observed = R_with_dpd.reindex(index=nodes)
corr = pd.Series(R_I_pred.values.flatten()).corr(pd.Series(observed.values.flatten()))
print(f"\nCorrelation between predicted R_I and observed R: {corr:.3f}")


In [ ]:
import numpy as np
import networkx as nx

def edges_from_matrix(r_df, threshold=1e-6):
    edges = []
    for target in r_df.index:          # rows = targets
        for source in r_df.columns:    # cols = sources
            w = r_df.loc[target, source]
            if source != target and np.isfinite(w) and abs(w) > threshold:
                edges.append((source, target, float(w)))
    return edges

EDGES = edges_from_matrix(rm_minus_inv, threshold=0.1)   # tune threshold

def build_graph():
    G = nx.DiGraph()
    for u, v, w in EDGES:
        G.add_edge(u, v, weight=w,
                   effect="activates" if w >= 0 else "inhibits",
                   sign=int(np.sign(w)))
    return G

G = build_graph()

for u, v, d in sorted(G.edges(data=True), key=lambda e: abs(e[2]["weight"]), reverse=True):
    print(f"{u:>8} -> {v:<8} {d['weight']:>10.3f}  ({d['effect']})")


SINK_NODES = ("DPD", "BTLA")

def draw(G, path):
    order = ["DPD", "BTLA", "JAK", "MTOR", "IGF1R", "NAE", "SERCA", "ARFGAP", "BCL"]
    order = [n for n in order if n in G] + [n for n in G if n not in order]
    theta = np.linspace(0, 2 * np.pi, len(order), endpoint=False)
    pos = {n: np.array([np.cos(t), np.sin(t)]) for n, t in zip(order, theta)}

    fig, ax = plt.subplots(figsize=(12, 10))
    dpd_w  = [abs(d["weight"]) for u, v, d in G.edges(data=True) if v in SINK_NODES]
    path_w = [abs(d["weight"]) for u, v, d in G.edges(data=True) if v not in SINK_NODES]

    def width(w, v):
        ref = max(dpd_w) if v in SINK_NODES else max(path_w)
        return 1.0 + 4.0 * (abs(w) / ref)

    nx.draw_networkx_nodes(G, pos, node_size=2200,
                           node_color="#e8eef7", edgecolors="#33475b", linewidths=1.5, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=[n for n in SINK_NODES if n in G], node_size=2800,
                           node_color="#ffe3b3", edgecolors="#b5651d", linewidths=2, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=11, font_weight="bold", ax=ax)

    for u, v, d in G.edges(data=True):
        color = "#2e7d32" if d["sign"] > 0 else "#c62828"
        style = "-|>" if d["sign"] > 0 else "-["   
        ax.annotate("", xy=pos[v], xytext=pos[u],
                    arrowprops=dict(arrowstyle=style, color=color,
                                    lw=width(d["weight"], v), alpha=0.8,
                                    shrinkA=22, shrinkB=22,
                                    connectionstyle="arc3,rad=0.12"))
        # weight label at edge midpoint
        mx, my = (pos[u] + pos[v]) / 2
        ax.text(mx, my, f"{d['weight']:.3g}", fontsize=7, color=color,
                ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))

    legend = [
        Line2D([0], [0], color="#2e7d32", lw=3, label="activates (+)"),
        Line2D([0], [0], color="#c62828", lw=3, label="inhibits (−)"),
    ]
    ax.legend(handles=legend, loc="upper left", frameon=True, fontsize=10)
    ax.set_title(f"Module interaction network  ({G.number_of_nodes()} nodes, "
                 f"{G.number_of_edges()} edges)", fontsize=13, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print(f"saved figure -> {path}")


if __name__ == "__main__":
    G = build_graph()
    print(f"Nodes: {G.number_of_nodes()}  ->  {sorted(G.nodes())}")
    print(f"Edges: {G.number_of_edges()}")
    print(f"Activating: {sum(1 for *_ , d in G.edges(data=True) if d['sign'] > 0)}, "
          f"Inhibiting: {sum(1 for *_ , d in G.edges(data=True) if d['sign'] < 0)}")
    print("\nIn-degree (targets):",  dict(G.in_degree()))
    print("Out-degree (sources):", dict(G.out_degree()))
    draw(G, "module_network_ols.png")

In [ ]:
r_ols_btla = rm_minus_inv

con_mat = pd.DataFrame(columns=['From','To','Strength'])
for con_to in r_ols_btla.index:
    for con_from in r_ols_btla.columns:
        if ((r_ols_btla.loc[con_to,con_from] != 0) & (con_from != con_to)):
            row_df = pd.DataFrame([[con_from,con_to,r_ols_btla.loc[con_to,con_from]]],columns=['From','To','Strength'])
            if con_mat.empty:
                con_mat = row_df.copy()
            else:
                con_mat = pd.concat([con_mat,row_df],axis=0,ignore_index=True)
            #con_mat = con_mat.append({'From':con_from,'To':con_to,'Strength':r_mean_df.loc[con_to,con_from]},ignore_index=True)
con_mat.to_csv('02_outputs/CD8_r_ols_btla_net.txt',sep='\t',index=False)
